# 🚀 AuthenticEye: Comprehensive Colab Training Suite

This notebook provides interactive, individual steps to train **all base models** (EfficientNet-B4, XceptionNet, ViT) and the **Fusion MLP Meta-Classifier** on Google Colab using a GPU runtime.

### Pre-requisites:
1. Ensure you have the `AuthenticEye-main` folder uploaded or cloned in your Google Drive.
2. Set Runtime to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU` or higher).

## 🛠️ Step 1: Mount Drive & Environment Setup

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Configure Path to your AuthenticEye Folder on Drive
PROJECT_ROOT = '/content/drive/MyDrive/AuthenticEye-main'
AI_SERVICE_DIR = os.path.join(PROJECT_ROOT, 'ai-service')

if not os.path.exists(AI_SERVICE_DIR):
    print(f"❌ ERROR: Could not find project at {AI_SERVICE_DIR}")
    print("Please verify the folder name in your Google Drive.")
else:
    os.chdir(AI_SERVICE_DIR)
    print(f"✅ Working directory set to: {os.getcwd()}")

# 3. Install core dependencies
print("📦 Installing dependencies...")
!pip install -q timm albumentations opencv-python-headless open_clip_torch datasets scikit-learn tqdm
!pip install -q -r requirements.txt

import torch
print(f"🔥 GPU STATUS: {'Enabled (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'DISABLED'}")

## 🔑 Step 2: Configure Kaggle API (Optional)
To download the standard `140k-real-and-fake-faces` deepfake dataset, enter your Kaggle API credentials below.

In [ ]:
import json
import os

# Enter your credentials
KAGGLE_USERNAME = ""
KAGGLE_KEY = ""

if KAGGLE_USERNAME and KAGGLE_KEY:
    kaggle_cred = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump(kaggle_cred, f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("✅ Kaggle credentials configured!")
else:
    print("⚠️ Missing credentials. You can also skip this and copy your own dataset into `ai-service/training/data/`.")

## 📂 Step 3: Dataset Preparation
This splits your source image folder into structured `train` and `val` directories. Change `--limit` to control the training subset size (leave blank for the full dataset).

In [ ]:
from scripts.prepare_dataset import main as prep_main

# If using Kaggle download, uncomment and modify:
# prep_main(['--source', './training/raw_data', '--dest', './training/prepared_data', '--download-kaggle', 'xhlulu/140k-real-and-fake-faces', '--limit', '1000'])

# Splitting the already uploaded local folder:
prep_main([
    '--source', './training/data',
    '--dest', './training/prepared_data',
    '--ratio', '0.8',
    '--limit', '2000'
])

## 🏋️ Step 4: Individual Base Model Training

### 1. Train EfficientNet-B4

In [ ]:
import shutil
from training.trainer import train_model
from types import SimpleNamespace

args = SimpleNamespace(
    model='efficientnet_b4',
    data_dir='./training/prepared_data',
    epochs=15,
    batch_size=32,
    lr=1e-4,
    loss='focal',
    use_amp=True,
    patience=10,
    checkpoint_dir='./checkpoints',
    log_dir='../tensorboard_logs',
    resume=None
)
train_model(args)

# Create copy matching standard model loader naming convention
shutil.copy2('./checkpoints/efficientnet_b4_best.pth', './checkpoints/efficientnet_b4.pth')


### 2. Train XceptionNet

In [ ]:
import shutil
from training.trainer import train_model
from types import SimpleNamespace

args = SimpleNamespace(
    model='xceptionnet',
    data_dir='./training/prepared_data',
    epochs=15,
    batch_size=32,
    lr=1e-4,
    loss='bce',
    use_amp=True,
    patience=10,
    checkpoint_dir='./checkpoints',
    log_dir='../tensorboard_logs',
    resume=None
)
train_model(args)

# Create copy matching standard model loader naming convention
shutil.copy2('./checkpoints/xceptionnet_best.pth', './checkpoints/xceptionnet.pth')


### 3. Train Vision Transformer (ViT)

In [ ]:
import shutil
from training.trainer import train_model
from types import SimpleNamespace

args = SimpleNamespace(
    model='vit',
    data_dir='./training/prepared_data',
    epochs=10,
    batch_size=16,
    lr=5e-5,
    loss='bce',
    use_amp=True,
    patience=10,
    checkpoint_dir='./checkpoints',
    log_dir='../tensorboard_logs',
    resume=None
)
train_model(args)

# Create copy matching standard model loader naming convention
shutil.copy2('./checkpoints/vit_best.pth', './checkpoints/vit.pth')


## 📊 Step 5: Feature Extraction for Fusion

In [ ]:
from scripts.extract_features import main as extract_main

# Runs the dataset through the trained base models alongside FFT/GAN/Diffusion signals
# to extract and save the 16-dimensional vector database.
extract_main([
    '--data_dir', './training/prepared_data',
    '--checkpoints_dir', './checkpoints',
    '--output_file', './checkpoints/extracted_features.pt'
])


## 🤖 Step 6: Train Fusion MLP Meta-Classifier

In [ ]:
from fusion.train_fusion import main as fusion_main

fusion_main([
    '--features_file', './checkpoints/extracted_features.pt',
    '--epochs', '50',
    '--save_path', './checkpoints/fusion_mlp.pth'
])


## 🏁 Step 7: Automated End-to-End Pipeline (Alternative)
Alternatively, you can trigger the entire pipeline (Prepare -> Train CNNs -> Extract Features -> Train Fusion) sequentially via a single script execution.

In [ ]:
from scripts.train_all_colab import main as master_main
import sys

sys.argv = [
    'train_all_colab.py',
    '--source_dir', './training/data',
    '--prepared_dir', './training/prepared_data',
    '--epochs_cnn', '15',
    '--epochs_vit', '10',
    '--epochs_fusion', '50',
    '--limit', '2000',
    '--batch_size', '32'
]
master_main()


## 📈 Step 8: Monitor Progress & Verify Checkpoints

In [ ]:
# 1. Monitor via TensorBoard
%load_ext tensorboard
%tensorboard --logdir ../tensorboard_logs

# 2. Verify all checkpoints can be loaded successfully by production API
import sys
sys.path.append(os.getcwd())
from model import load_models
load_models()